In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/usgs_inventory_missing_cleaned.csv")

print("Cleaned dataset loaded successfully!")
print("Shape:", df.shape)

Cleaned dataset loaded successfully!
Shape: (8981, 37)


In [2]:
# Cell 18: Test wildfire-grouped cross-validation

from sklearn.model_selection import StratifiedGroupKFold

X = df.drop(columns=["Response"])
y = df["Response"]
groups = df["FireName"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups), start=1
):

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    overlap = train_groups.intersection(test_groups)

    fold_results.append({
        "Fold": fold,
        "Train_Rows": len(train_idx),
        "Test_Rows": len(test_idx),
        "Test_Positive": int(y.iloc[test_idx].sum()),
        "Test_Negative": int((y.iloc[test_idx] == 0).sum()),
        "Test_Wildfires": len(test_groups),
        "Overlapping_Wildfires": len(overlap)
    })

fold_results = pd.DataFrame(fold_results)

fold_results

,Fold,Train_Rows,Test_Rows,Test_Positive,Test_Negative,Test_Wildfires,Overlapping_Wildfires
0,1,6247,2734,217,2517,13,0
1,2,7038,1943,221,1722,16,0
2,3,7404,1577,220,1357,17,0
3,4,7566,1415,222,1193,15,0
4,5,7669,1312,260,1052,19,0


In [3]:
# Define target and wildfire grouping column

target = "Response"
group_column = "FireName"

print("Target variable:", target)
print("Grouping variable:", group_column)

Target variable: Response
Grouping variable: FireName


In [4]:
# Cell 23: Import modelling libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

# Five-fold validation, keeping each wildfire in only one fold
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation strategy:", cv)
print("Number of folds: 5")
print("Grouping variable:", group_column)

Cross-validation strategy: StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
Number of folds: 5
Grouping variable: FireName


In [5]:
# Cell 24: Define machine-learning models

models = {
    "Dummy Baseline": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=10,
        min_samples_leaf=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance"
    ),

    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

print("Models defined:")
for name in models:
    print("-", name)


Models defined:
- Dummy Baseline
- Logistic Regression
- Decision Tree
- Random Forest
- KNN
- SVM


In [6]:
# Define target and wildfire grouping column

target = "Response"
group_column = "FireName"

# Full feature set
full_features = [
    "StormAccumulation",
    "StormDuration",
    "PeakI15",
    "PeakI30",
    "PeakI60",
    "1YearStormI15",
    "1YearStormI30",
    "1YearStormI60",
    "RainfallRatio15",
    "RainfallRatio30",
    "RainfallRatio60",
    "HM23",
    "dNBR/1000",
    "KF",
    "UpstreamArea",
    "DaysSinceFire",
    "YearAfterFire"
]

# Reduced feature set
reduced_features = [
    "StormAccumulation",
    "StormDuration",
    "PeakI60",
    "RainfallRatio60",
    "HM23",
    "dNBR/1000",
    "KF",
    "UpstreamArea",
    "DaysSinceFire",
    "YearAfterFire"
]

# Prepare target and wildfire groups
y = df[target].copy()
groups = df[group_column].copy()

print("Full features:", len(full_features))
print("Reduced features:", len(reduced_features))
print("Target:", target)
print("Wildfire groups:", groups.nunique())

Full features: 17
Reduced features: 10
Target: Response
Wildfire groups: 80


In [7]:
# Cell 25: Evaluate full and reduced feature sets

feature_sets = {
    "Full Features": full_features,
    "Reduced Features": reduced_features
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

results = []

for feature_set_name, feature_list in feature_sets.items():

    X_current = df[feature_list].copy()

    for model_name, model in models.items():

        # Preprocessing is fitted independently within each training fold
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        scores = cross_validate(
            pipeline,
            X_current,
            y,
            groups=groups,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False,
            error_score="raise"
        )

        results.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "Accuracy": scores["test_accuracy"].mean(),
            "Precision": scores["test_precision"].mean(),
            "Recall": scores["test_recall"].mean(),
            "F1": scores["test_f1"].mean(),
            "ROC_AUC": scores["test_roc_auc"].mean(),
            "PR_AUC": scores["test_pr_auc"].mean()
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["PR_AUC", "Recall"],
    ascending=False
).reset_index(drop=True)

print("MODEL COMPARISON RESULTS")
display(results_df.round(4))


MODEL COMPARISON RESULTS


,Feature_Set,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Reduced Features,Logistic Regression,0.7808,0.3662,0.7592,0.4901,0.8395,0.4906
1,Full Features,Random Forest,0.8521,0.5414,0.3330,0.3786,0.8511,0.4790
2,Full Features,Logistic Regression,0.7645,0.3253,0.6719,0.4320,0.8274,0.4630
3,Reduced Features,Random Forest,0.8516,0.4532,0.2341,0.2971,0.8443,0.4535
4,Full Features,SVM,0.7868,0.4043,0.7741,0.5158,0.8438,0.4495
5,Full Features,KNN,0.8559,0.5404,0.3201,0.3792,0.7957,0.4072
6,Reduced Features,SVM,0.7617,0.3443,0.7545,0.4657,0.8069,0.3865
7,Reduced Features,KNN,0.8526,0.4615,0.3025,0.3517,0.7816,0.3793
8,Full Features,Decision Tree,0.7668,0.3596,0.6342,0.4473,0.7178,0.3420
9,Reduced Features,Decision Tree,0.7166,0.2850,0.6409,0.3830,0.6781,0.2893
